# Stage 10 — DINOv2 last-2-blocks unfreeze + 7x7 fingertip-window (1-fold gate)

Binary gate test: **does fine-tuning DINOv2 around the fingertip fundamentally change the appearance-vs-kinematics story?**

Pass criterion (per the post-Task-A prompt §4):
- ✅ **Transform** (≥ 0.20 CER drop vs Stage 9a fold-0 = 0.5681 → target ≤ 0.368): reopen the V-L matrix, run full 5-fold.
- 🟡 **Mild** (0.05 ≤ ΔCER < 0.20): document and move on; expensive to scale.
- ❌ **Nudge** (ΔCER < 0.05): drop V-L permanently from the dissertation roadmap.

## Design (pragmatic, Kaggle-T4-fitted)

| Knob | Value | vs Stage 3 / Stage 9a |
|---|---|---|
| DINOv2 backbone | dinov2-small at 336x336 | same as Stage 3 |
| **Unfreeze** | **last 2 of 12 blocks** | new (Stage 3 = frozen) |
| Window K (radius) | 3 (7x7 patches) | bigger than Stage 3's 3x3 |
| Bell-pool sigma | 1.5 | wider for 7x7 |
| ±1 temporal context + visibility gate | yes | same as Stage 3 |
| Conformer + joint CTC+attn | yes | same as Stage 9a |
| LR (head) | 5e-4 | Stage 9a |
| LR (DINOv2 unfrozen) | 5e-6 | new |
| Fold | 0 only (val signers HJH, KIS, KJM, LSB, OSW, PJH, RJH, YJH) | Stage 9a fold 0 |
| Training clips | **500 random fold-0 train signers' clips** (subset for RAM) | Stage 9a uses full 2757 |
| Val clips | full fold-0 val (720 clips) | full |
| Epochs | 60 | Stage 9a uses 80 |
| Batch | 16 | Stage 9a uses 32 (memory-limited here) |

**Wall-clock estimate on Kaggle T4 (one session)**:

| Phase | Time |
|---|---|
| Stream fold-0 clips + MediaPipe + crop+resize into RAM cache | ~30 min |
| 60 epochs × ~1.5 min/epoch (500 train clips, batch 16, last 2 blocks backward) | ~90 min |
| Per-epoch val (720 clips, forward only) | ~20 min total |
| Total | **~2 h 20 min** |

Fits one Kaggle session with margin.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy transformers --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate manifest (skeleton cache optional)

In [ ]:
import os, glob, json
def _first(pat):
    m = (glob.glob(f'/kaggle/working/**/{pat}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pat}',  recursive=True))
    return m[0] if m else None

CV_MANIFEST = _first('subject_cv5.json')
print(f'manifest: {CV_MANIFEST}  (exists={bool(CV_MANIFEST)})')
assert CV_MANIFEST, 'subject_cv5.json must be available (attach a kernel that has it)'

## Cell 3 — Config

In [ ]:
import random, logging
import numpy as np
from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

LOG_DIR  = '/kaggle/working/logs'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage10.log'))])

ABLATION_FOLD     = 0
N_TRAIN_SUBSET    = 500     # cap fold-0 train at 500 random clips (RAM)
SEED              = 42
VARIANT_NAME      = 'stage10_d2u2_w3'

# Locked from Stage 9a except where overridden.
T_NATIVE     = 32
IMAGE_SIZE   = 336
WINDOW_K     = 3
BELL_SIGMA   = 1.5
UNFREEZE_LAST = 2
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 16
NUM_EPOCHS   = 60
WARMUP_PCT   = 0.05
LR_HEAD      = 5e-4
LR_DINOV2    = 5e-6
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
DEC_N_LAYERS = 2          # Stage 9a ablation winner
DEC_N_HEADS  = 4
LAMBDA_CTC   = 0.3

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
                      lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
                      grad_clip=GRAD_CLIP, num_workers=2,
                      warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device          : {cfg.device}')
print(f'Variant         : {VARIANT_NAME}')
print(f'Unfreeze        : last {UNFREEZE_LAST} DINOv2 blocks')
print(f'Window K        : {WINDOW_K}  (7x7 patches)')
print(f'Train clips cap : {N_TRAIN_SUBSET}  (subset of fold-{ABLATION_FOLD})')

## Cell 4 — Stream fold-0 clips and pre-extract the in-RAM frame cache

Streams the full WiTA English subset, filters to fold-0 train + val signers, runs MediaPipe + bbox crop + resize.  ~30 min on T4.

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
from wita_v2.datasets.cv_splits       import load_cv5_manifest, fold_indices
from wita_v2.datasets.skeleton_cache  import LandmarkExtractor
from wita_v2.training.stage10_train   import _Stage10ClipCache

manifest = load_cv5_manifest(CV_MANIFEST)
train_subjects = set(manifest['folds'][ABLATION_FOLD]['train_subjects'])
val_subjects   = set(manifest['folds'][ABLATION_FOLD]['val_subjects'])
print(f'fold {ABLATION_FOLD}: train signers={len(train_subjects)}  val signers={len(val_subjects)}')

all_samples = stream_and_index_with_subjects(cfg)
fold_samples = [s for s in all_samples if s[2] in train_subjects or s[2] in val_subjects]
print(f'fold-0 clips streamed: {len(fold_samples)}')
del all_samples

# Random subset of train signers' clips.
train_pool = [s for s in fold_samples if s[2] in train_subjects]
val_pool   = [s for s in fold_samples if s[2] in val_subjects]
rng = random.Random(SEED)
rng.shuffle(train_pool)
train_pool = train_pool[:N_TRAIN_SUBSET]
print(f'train pool (subset): {len(train_pool)}  val pool (full): {len(val_pool)}')

extractor = LandmarkExtractor()
train_cache = _Stage10ClipCache(image_size=IMAGE_SIZE, T_native=T_NATIVE)
val_cache   = _Stage10ClipCache(image_size=IMAGE_SIZE, T_native=T_NATIVE)
import time
t0 = time.time()
for i, (fb, lab, subj) in enumerate(train_pool):
    train_cache.add_clip(fb, lab, subj, extractor)
    if (i + 1) % 50 == 0:
        print(f'  train {i+1}/{len(train_pool)}  cache={train_cache.memory_mb/1024:.1f} GB')
for i, (fb, lab, subj) in enumerate(val_pool):
    val_cache.add_clip(fb, lab, subj, extractor)
    if (i + 1) % 100 == 0:
        print(f'  val {i+1}/{len(val_pool)}  cache={val_cache.memory_mb/1024:.1f} GB')
extractor.close()

print(f'\ntrain cache: {len(train_cache)} clips, {train_cache.memory_mb/1024:.2f} GB')
print(f'val cache  : {len(val_cache)} clips, {val_cache.memory_mb/1024:.2f} GB')
print(f'cache build time: {(time.time()-t0)/60:.1f} min')

# Free the raw bytes lists.
del train_pool, val_pool, fold_samples

## Cell 5 — Train (end-to-end, last 2 DINOv2 blocks unfrozen)

In [ ]:
from wita_v2.training.stage10_train import train_stage10
result = train_stage10(
    train_clip_cache=train_cache, val_clip_cache=val_cache,
    cfg=cfg, fold=ABLATION_FOLD, variant=VARIANT_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, grad_clip=GRAD_CLIP,
    dropout=DROPOUT, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
    dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
    lambda_ctc=LAMBDA_CTC, unfreeze_last_n=UNFREEZE_LAST,
    window_k=WINDOW_K, bell_sigma=BELL_SIGMA,
    lr_dinov2=LR_DINOV2, lr_head=LR_HEAD, weight_decay=WEIGHT_DECAY,
    seg_chunk=16, seed=SEED,
    checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
)
RESULTS_PATH = '/kaggle/working/stage10_results.json'
summary = {k: v for k, v in result.items() if k != 'history'}
with open(RESULTS_PATH, 'w') as f:
    json.dump([summary], f, indent=2)
print(f'\nWrote {RESULTS_PATH}')

## Cell 6 — Gate verdict

In [ ]:
STAGE9A_FOLD0 = 0.5681
delta = STAGE9A_FOLD0 - result['best_val_cer']
print(f'Stage 9a fold-0 baseline : {STAGE9A_FOLD0:.4f}')
print(f'Stage 10 fold-0 (this)   : {result["best_val_cer"]:.4f}  (best epoch {result["best_epoch"]})')
print(f'ΔCER (Stage 10 wins by)  : {delta:+.4f}')
print()
if delta >= 0.20:
    print(f'  ✅ TRANSFORM (ΔCER ≥ 0.20) — DINOv2 unfreeze fundamentally helps.')
    print('     Reopen the V-L matrix.  Run full 5-fold (Colab Pro A100 recommended).')
elif delta >= 0.05:
    print(f'  🟡 MILD ({0.05:.2f} ≤ ΔCER < 0.20) — unfreeze nudges but expensive to scale.')
    print('     Document in the appendix; consider 5-fold ONLY if dissertation timeline allows.')
elif delta >= -0.01:
    print(f'  ❌ NUDGE (ΔCER < 0.05) — DINOv2 features do not carry the kinematic signal')
    print('     even when fine-tuned.  Drop V-L permanently from the dissertation.')
else:
    print(f'  ⚠️  REGRESS — unfreeze destabilises training.  Check unfreeze_last_n / LR.')

## Cell 7 — Per-signer table for fold 0

In [ ]:
import numpy as np
STAGE9A_FOLD0_PS = {  # Stage 9a no_dann fold-0 per-signer (from logs)
  'HJH': 0.5336, 'KIS': 0.3345, 'KJM': 0.7027, 'LSB': 0.5572,
  'RJH': 0.5621, 'OSW': 0.6619, 'PJH': 0.8237, 'YJH': 0.3671,
}
print(' signer    Stage 9a    Stage 10    ΔCER')
for s, v9 in sorted(STAGE9A_FOLD0_PS.items(), key=lambda kv: kv[1]):
    v10 = result['best_per_signer_val_cer'].get(s, float('nan'))
    d = v9 - v10
    print(f'  {s}      {v9:.4f}      {v10:.4f}      {d:+.4f}')

## Cell 8 — Commit kernel

Save Version → Save & Run All to preserve:
- `stage10_results.json`
- `checkpoints/stage10_fold0_stage10_d2u2_w3_best.pt`  (dinov2 + encoder + decoder state dicts)
- `logs/stage10.log`
- `logs/stage10_fold0_stage10_d2u2_w3_history.json` + `_full.json`